# Trening modelu

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [2]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# TWÓJ KOD
# 1. train_test_split (80/20, stratify=y)
# 2. RandomForestClassifier(100)
# 3. classification_report
# 4. pickle.dump do 'fraud_model.pkl'

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pickle

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Raport klasyfikacji:")
print(classification_report(y_test, y_pred))

with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model został zapisany pomyślnie.")

Raport klasyfikacji:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model został zapisany pomyślnie.


# FastAPI

In [4]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

# -----

@app.post("/score")
async def score_transaction(tx: Transaction):
    data = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    
    prediction = model.predict(data)[0]
    
    probability = model.predict_proba(data)[0][1]
    
    return {
        "is_fraud": bool(prediction),
        "fraud_probability": float(probability)
    }

Writing fraud_api.py


In [8]:
import requests
podejrzana_data = {
    "amount": 5500, 
    "is_electronics": 1, 
    "tx_per_minute": 12
}

r_fraud = requests.post("http://localhost:8001/score", json=podejrzana_data)

print("Podejrzana:", r_fraud.json())

Podejrzana: {'is_fraud': True, 'fraud_probability': 0.99}


# Kafka + ML

In [9]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json
import requests

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

print("Serwis scoringowy ML uruchomiony...")

for message in consumer:
    tx = message.value
    
    is_electronics = 1 if tx.get('category') == 'elektronika' else 0
    
    features = {
        "amount": float(tx.get('amount', 0)),
        "is_electronics": int(is_electronics),
        "tx_per_minute": int(tx.get('tx_per_minute', 5))
    }
    
    try:
        response = requests.post(API_URL, json=features)
        prediction = response.json()
        
        if prediction.get("is_fraud"):
            alert_data = {
                "timestamp": datetime.now().isoformat(),
                "tx_id": tx.get("tx_id"),
                "fraud_probability": prediction.get("fraud_probability"),
                "original_data": tx
            }
            
            alert_producer.send('alerts', value=alert_data)
            
            print(f"!!! ALERT !!! Wykryto oszustwo! ID: {tx.get('tx_id')} | "
                  f"Prawdopodobieństwo: {prediction.get('fraud_probability'):.2%}")
            
    except Exception as e:
        print(f"Błąd podczas scoringu transakcji {tx.get('tx_id')}: {e}")

alert_producer.flush()

Writing ml_consumer.py
